Building a REST API using FastAPI that provides product recommendations based on user preferences using a simple recommendation logic (content-based or similarity-based).<br>
This task simulates how real-world AI systems expose models as APIs (like Amazon, Netflix, etc.).


In [1]:
# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

# Data Handling
import pandas as pd
import numpy as np

# Text Processing
import re

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Model Saving
import pickle

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================================
# STEP 2: LOAD DATASET
# ============================================================

# Load Dataset
df = pd.read_csv("./app/data/products.csv")

# Display First 5 Rows
df.head()

,name,main_category,sub_category,image,link
0,NEUTRON Fashionable Analog Brown and Black Col...,accessories,Watches,https://m.media-amazon.com/images/I/717k6KrsGJ...,https://www.amazon.in/NEUTRON-Fashionable-Anal...
1,Sonata Analog Off White Dial Women's Watch-816...,accessories,Watches,https://m.media-amazon.com/images/I/61ioBO3yIx...,https://www.amazon.in/Sonata-Analog-White-Wome...
2,Prolet 20MM Silicone Strap Compatible with Ama...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Prolet-Silicone-Compatib...
3,M.G.R.J® Pro HD+ Tempered Glass Screen Protect...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/M-G-R-J-Apple-iPad-6th-G...
4,Fastrack Analog Yellow Dial Women's Watch-6230...,accessories,Watches,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Fastrack-Analog-Yellow-W...


In [3]:
# ============================================================
# STEP 3: DATA EXPLORATION
# ============================================================

# Dataset Shape
print("Dataset Shape:", df.shape)

# Column Names
print("\nColumns:")
print(df.columns)

# Check Missing Values
print("\nMissing Values:")
print(df.isnull().sum())

# Check Duplicate Rows
print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (28733, 5)

Columns:
Index(['name', 'main_category', 'sub_category', 'image', 'link'], dtype='object')

Missing Values:
name             0
main_category    0
sub_category     0
image            0
link             0
dtype: int64

Duplicate Rows: 0


In [4]:
# ============================================================
# STEP 4: DATA CLEANING
# ============================================================

# Remove Missing Values
df.dropna(inplace=True)

# Remove Duplicate Products
df.drop_duplicates(inplace=True)

# Reset Index
df.reset_index(drop=True, inplace=True)

print("Data Cleaned Successfully")
print("New Shape:", df.shape)

Data Cleaned Successfully
New Shape: (28733, 5)


In [5]:
# ============================================================
# STEP 5: SELECT IMPORTANT COLUMNS
# ============================================================

# Keep Only Important Columns
df = df[['name', 'main_category', 'sub_category', 'image', 'link']]

# Display Sample Data
df.head()

,name,main_category,sub_category,image,link
0,NEUTRON Fashionable Analog Brown and Black Col...,accessories,Watches,https://m.media-amazon.com/images/I/717k6KrsGJ...,https://www.amazon.in/NEUTRON-Fashionable-Anal...
1,Sonata Analog Off White Dial Women's Watch-816...,accessories,Watches,https://m.media-amazon.com/images/I/61ioBO3yIx...,https://www.amazon.in/Sonata-Analog-White-Wome...
2,Prolet 20MM Silicone Strap Compatible with Ama...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Prolet-Silicone-Compatib...
3,M.G.R.J® Pro HD+ Tempered Glass Screen Protect...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/M-G-R-J-Apple-iPad-6th-G...
4,Fastrack Analog Yellow Dial Women's Watch-6230...,accessories,Watches,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Fastrack-Analog-Yellow-W...


In [6]:
# ============================================================
# STEP 6: TEXT PREPROCESSING
# ============================================================

# Function to Clean Text
def clean_text(text):
    
    # Convert to Lowercase
    text = text.lower()
    
    # Remove Special Characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Remove Extra Spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply Cleaning
df['clean_name'] = df['name'].apply(clean_text)
df['clean_main_category'] = df['main_category'].apply(clean_text)
df['clean_sub_category'] = df['sub_category'].apply(clean_text)

print("Text Cleaning Completed")

Text Cleaning Completed


In [7]:
# ============================================================
# STEP 7: FEATURE ENGINEERING
# ============================================================

# Combine Important Features
df['combined_features'] = (
    df['clean_name'] + ' ' +
    df['clean_main_category'] + ' ' +
    df['clean_sub_category']
)

# Display Combined Features
df[['name', 'combined_features']].head()

,name,combined_features
0,NEUTRON Fashionable Analog Brown and Black Col...,neutron fashionable analog brown and black col...
1,Sonata Analog Off White Dial Women's Watch-816...,sonata analog off white dial womens watch8169y...
2,Prolet 20MM Silicone Strap Compatible with Ama...,prolet 20mm silicone strap compatible with ama...
3,M.G.R.J® Pro HD+ Tempered Glass Screen Protect...,mgrj pro hd tempered glass screen protector co...
4,Fastrack Analog Yellow Dial Women's Watch-6230...,fastrack analog yellow dial womens watch6230wl...


In [8]:
# ============================================================
# STEP 8: TF-IDF VECTORIZATION
# ============================================================

# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Convert Text into Numerical Vectors
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (28733, 27947)


In [9]:
# ============================================================
# STEP 9: CALCULATE COSINE SIMILARITY
# ============================================================

# Compute Similarity Matrix
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (28733, 28733)


In [10]:
# ============================================================
# STEP 10: PRODUCT SEARCH FUNCTION
# ============================================================

def search_product(product_name):
    
    # Convert Input to Lowercase
    product_name = product_name.lower()
    
    # Search Matching Products
    matches = df[df['name'].str.lower().str.contains(product_name)]
    
    # Return Top 10 Matches
    return matches[['name', 'main_category']].head(10)

# Example
search_product("watch")

,name,main_category
0,NEUTRON Fashionable Analog Brown and Black Col...,accessories
1,Sonata Analog Off White Dial Women's Watch-816...,accessories
4,Fastrack Analog Yellow Dial Women's Watch-6230...,accessories
5,NEUTRON Fancy Analog Rose Gold Color Dial Girl...,accessories
6,Sonata Epic Gents Analog Watch - EP10002NL01,accessories
7,GROOT Professional Analog Red and Black Color ...,accessories
10,NEUTRON Quartz Analog Black and Silver Color D...,accessories
11,Silicone Waterproof Replacement Watch Straps B...,"tv, audio & cameras"
12,Relish Analogue Girl's Watch (Black Dial Pink ...,accessories
13,TIMENTER Wrist Analog Gold and Black Color Dia...,accessories


In [11]:
# ============================================================
# STEP 11: PRODUCT RECOMMENDATION FUNCTION
# ============================================================

def recommend_products(product_name, top_n=5):
    
    # Convert Input to Lowercase
    product_name = product_name.lower()
    
    # Find Product Index
    matches = df[df['name'].str.lower().str.contains(product_name)]
    
    # Handle Invalid Product
    if matches.empty:
        return "Product Not Found"
    
    # Get First Matching Product Index
    product_index = matches.index[0]
    
    # Get Similarity Scores
    similarity_scores = list(enumerate(similarity_matrix[product_index]))
    
    # Sort Products Based on Similarity
    sorted_products = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    # Remove First Product (Same Product)
    sorted_products = sorted_products[1:top_n+1]
    
    # Store Recommendations
    recommendations = []
    
    for index, score in sorted_products:
        
        recommendations.append({
            'Product Name': df.iloc[index]['name'],
            'Category': df.iloc[index]['main_category'],
            'Similarity Score': round(score, 2),
            'Image': df.iloc[index]['image'],
            'Product Link': df.iloc[index]['link']
        })
    
    return pd.DataFrame(recommendations)

# Example
recommend_products("Fire-Boltt")

,Product Name,Category,Similarity Score,Image,Product Link
0,"Fire-Boltt Dagger 1.43"" AMOLED Display Smartwa...","tv, audio & cameras",1.00,https://m.media-amazon.com/images/I/61ylgwLi2P...,https://www.amazon.in/Fire-Boltt-Smartwatch-Bl...
1,"Fire-Boltt Dagger 1.43"" AMOLED Display Smartwa...","tv, audio & cameras",1.00,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Fire-Boltt-Smartwatch-Bl...
2,"Fire-Boltt Invincible Plus 1.43"" AMOLED Displa...","tv, audio & cameras",0.38,https://m.media-amazon.com/images/I/61GJzEhRrs...,https://www.amazon.in/Fire-Boltt-Invincible-Sm...
3,"Fire-Boltt Invincible Plus 1.43"" AMOLED Displa...","tv, audio & cameras",0.38,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Fire-Boltt-Invincible-Sm...
4,"Fire-Boltt Invincible Plus 1.43"" AMOLED Displa...","tv, audio & cameras",0.38,https://m.media-amazon.com/images/I/61raZLNoTz...,https://www.amazon.in/Fire-Boltt-Invincible-Sm...


In [12]:
# ============================================================
# STEP 12: USER PREFERENCE RECOMMENDATION
# ============================================================

def recommend_by_preferences(preferences, top_n=5):
    
    # Clean User Input
    preferences = clean_text(preferences)
    
    # Convert User Input into Vector
    user_vector = tfidf.transform([preferences])
    
    # Compute Similarity
    scores = cosine_similarity(user_vector, tfidf_matrix)
    
    # Get Top Products
    top_indices = scores[0].argsort()[::-1][:top_n]
    
    # Store Results
    results = []
    
    for index in top_indices:
        
        results.append({
            'Product Name': df.iloc[index]['name'],
            'Category': df.iloc[index]['main_category'],
            'Score': round(scores[0][index], 2),
            'Image': df.iloc[index]['image'],
            'Product Link': df.iloc[index]['link']
        })
    
    return pd.DataFrame(results)

# Example
recommend_by_preferences("bluetooth smartwatch electronics")

,Product Name,Category,Score,Image,Product Link
0,Smartwatch,accessories,0.72,https://m.media-amazon.com/images/I/41VWDTCJ0F...,https://www.amazon.in/Generic-Smartwatch/dp/B0...
1,Biggy smartwatch for Women Women's Rose Gold S...,accessories,0.48,https://m.media-amazon.com/images/I/613Zb46AWw...,https://www.amazon.in/Biggy-smartwatch-Smartwa...
2,Biggy smartwatch for Women Women's Rose Gold S...,accessories,0.48,https://m.media-amazon.com/images/I/613Zb46AWw...,https://www.amazon.in/Biggy-smartwatch-Smartwa...
3,Fire-Boltt India's No 1 Smartwatch Brand Talk ...,"tv, audio & cameras",0.46,https://m.media-amazon.com/images/I/61NTshPHLm...,https://www.amazon.in/Fire-Boltt-Smartwatch-Bl...
4,Fire-Boltt India's No 1 Smartwatch Brand Talk ...,"tv, audio & cameras",0.46,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/Fire-Boltt-Smartwatch-Bl...


In [13]:
# ============================================================
# STEP 13: USER SEARCH HISTORY TRACKING
# ============================================================

# Create Empty Search History List
search_history = []

def track_search(product_name):
    
    # Add Search to History
    search_history.append(product_name)
    
    print(f"'{product_name}' added to search history")

# Example
track_search("smartwatch")

# Display History
search_history

'smartwatch' added to search history


['smartwatch']

In [15]:
# ============================================================
# STEP 14: SAVE MODEL USING PICKLE
# ============================================================

# Save TF-IDF Vectorizer
with open("./app/model/tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

# Save Similarity Matrix
with open("./app/model/similarity_matrix.pkl", "wb") as file:
    pickle.dump(similarity_matrix, file)

# Save Dataframe
with open("./app/model/products_dataframe.pkl", "wb") as file:
    pickle.dump(df, file)

print("Models Saved Successfully")

Models Saved Successfully


In [16]:
# ============================================================
# STEP 15: LOAD SAVED FILES
# ============================================================

# Load TF-IDF Vectorizer
with open("./app/model/tfidf_vectorizer.pkl", "rb") as file:
    loaded_tfidf = pickle.load(file)

# Load Similarity Matrix
with open("./app/model/similarity_matrix.pkl", "rb") as file:
    loaded_similarity = pickle.load(file)

# Load Dataframe
with open("./app/model/products_dataframe.pkl", "rb") as file:
    loaded_df = pickle.load(file)

print("Saved Files Loaded Successfully")

Saved Files Loaded Successfully


In [17]:
# ============================================================
# STEP 16: FINAL TESTING
# ============================================================

# Test Recommendations
recommend_products("Marshall")

,Product Name,Category,Similarity Score,Image,Product Link
0,Marshall Willen Portable Bluetooth Speaker - B...,"tv, audio & cameras",0.60,https://m.media-amazon.com/images/I/61zYC5Duim...,https://www.amazon.in/Marshall-Willen-Portable...
1,M.G.R.J® Portable Carrying Case Cover for Mars...,"tv, audio & cameras",0.53,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/M-G-R-J%C2%AE-Portable-C...
2,boAt PartyPal 20 15 Watt Wireless Bluetooth Pa...,"tv, audio & cameras",0.44,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/PartyPal-20-Integrated-M...
3,10WeRun 10 Watt Wireless Bluetooth Portable Ou...,"tv, audio & cameras",0.37,https://m.media-amazon.com/images/W/IMAGERENDE...,https://www.amazon.in/10WeRun-Portable-Bluetoo...
4,MorningVale Wireless Bluetooth Portable Speake...,"tv, audio & cameras",0.36,https://m.media-amazon.com/images/I/61JUEwg23K...,https://www.amazon.in/XenderCage-Bluetooth-Mic...
